# Tree-Based Models

Unlike the previous notebook, this stage of the project is devided into two separate notebooks based on the preprocessing pipeline required by each model. RandomForest and XGBoost, both rely on the same preprocessing approach and are therefore evaluated together in this notebook. CatBoost, however, uses a different preprocessing strategy, by handling categorical features natively and is evaluated separately in the next notebook.

Raandom Forest serves as the baseline tree-based model for both notebooks. Its evaluated metrics are saved to a CSV file so they can be reused in the CatBoost notebook, allowing all tree-based models to be compared consistently without retraining the baseline model.

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import sys
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 500)

sys.path.append(str(Path().cwd().parent.resolve()))

import preprocessing.features as features

builder = features.FeatureBuilder()

df = pd.read_csv(features.DATASET_PATH)

df_copy = builder.get_df(df)

df_copy.shape

/home/carl/notebooks/airbnb_prices_prediction/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(74111, 27)

In [2]:
import xgboost

xgboost.__version__

'3.3.0'

In [3]:
import sklearn

sklearn.__version__

'1.9.0'

# Baseline Model. RandomForest

In [4]:
df_copy = builder.get_df(df, use_amenities=False, use_embeddings=False)

X = df_copy.drop(columns=['log_price'])
y = df_copy['log_price']

X = X.drop(columns=['zipcode'])

X.shape, y.shape

((74111, 25), (74111,))

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    random_state=42,
    test_size=0.2
)

X_train.shape, X_test.shape

((59288, 25), (14823, 25))

In [6]:
import preprocessing.tree_preprocessor as preprocessor
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

cat_features = X_train.select_dtypes(include=['string', 'object']).columns

tree_preprocessor = preprocessor.create_tree_preprocessor(cat_features)

tree_pipeline = Pipeline([
    ("preprocessor", tree_preprocessor),
    ("model", RandomForestRegressor(random_state=42, n_jobs=-1))
])

tree_pipeline.fit(X_train, y_train)

y_pred_test_log = tree_pipeline.predict(X_test)
y_pred_train_log = tree_pipeline.predict(X_train)

y_pred_test = np.exp(y_pred_test_log)
y_pred_train = np.exp(y_pred_train_log)

test_MAE = mean_absolute_error(np.exp(y_test), y_pred_test)
train_MAE = mean_absolute_error(np.exp(y_train), y_pred_train)

test_RMSE = root_mean_squared_error(np.exp(y_test), y_pred_test)
train_RMSE = root_mean_squared_error(np.exp(y_train), y_pred_train)

test_r2 = r2_score(y_test, y_pred_test_log)
train_r2 = r2_score(y_train, y_pred_train_log)

print(f"Test MAE: {test_MAE:.2f}$ | Train MAE: {train_MAE:.2f}$")
print(f"Test RMSE: {test_RMSE:.2f}$ | Train RMSE: {train_RMSE:.2f}$")
print(f"Test R2 Score: {test_r2:.2f} | Train R2 Score: {train_r2:.2f}")

Test MAE: 52.22$ | Train MAE: 21.08$
Test RMSE: 118.45$ | Train RMSE: 56.19$
Test R2 Score: 0.68 | Train R2 Score: 0.95


## RandomForest Conclusion

The baseline RandomForest model exhibit significant overfitting, with substantially better performance on the training set that on the test set. Therefore, its current evaluation metrics are not suitable as the primary baseline for comparing tree-based models. In the next step, hyperparameter tuning will be performed using RandomizedSearchCV to reduce overfitting and establish a more reliable baseline for subsequent comparisons. 

# RandomForest. Hyperparameter Tuning.

In [7]:
ARTIFACTS_DIR = Path('../artifacts')
ARTIFACTS_DIR.mkdir(exist_ok=True)

BASELINE_RANDOM_SEARCH_MODEL = ARTIFACTS_DIR / "random_forest_random_search.joblib"

In [8]:
%%time

import joblib
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

if BASELINE_RANDOM_SEARCH_MODEL.exists():
    print("Loading RandomizedSearchCV...")
    random_search = joblib.load(BASELINE_RANDOM_SEARCH_MODEL)
else:
    print("Training RandomizedSearchCV...")
    
    tree_pipeline = Pipeline([
        ("preprocessor", tree_preprocessor),
        ("model", RandomForestRegressor(random_state=42))
    ])
    
    param_dist = {
        "model__n_estimators": randint(150, 600),
        "model__max_depth": [10, 15, 20, 25, 30],
        "model__min_samples_split": randint(10, 40),
        "model__min_samples_leaf": randint(3, 15),
        "model__max_features": ['log2', 'sqrt', 0.3, 0.3],
        "model__ccp_alpha": [0.0, 0.0001, 0.0005, 0.001]
    }
    
    random_search = RandomizedSearchCV(
        estimator=tree_pipeline,
        param_distributions=param_dist,
        n_iter=15,
        scoring="neg_root_mean_squared_error",
        cv=3,
        random_state=42,
        n_jobs=-1,
        verbose=2
    )

    random_search.fit(X_train, y_train)
    joblib.dump(random_search, BASELINE_RANDOM_SEARCH_MODEL)

best_rf = random_search.best_estimator_
random_search.best_params_

Loading RandomizedSearchCV...
CPU times: user 50.9 ms, sys: 53.4 ms, total: 104 ms
Wall time: 104 ms


{'model__ccp_alpha': 0.0,
 'model__max_depth': 25,
 'model__max_features': 0.3,
 'model__min_samples_leaf': 10,
 'model__min_samples_split': 12,
 'model__n_estimators': 299}

In [9]:
y_pred_test_log = best_rf.predict(X_test)
y_pred_train_log = best_rf.predict(X_train)

y_pred_test = np.exp(y_pred_test_log)
y_pred_train = np.exp(y_pred_train_log)

test_MAE = mean_absolute_error(np.exp(y_test), y_pred_test)
train_MAE = mean_absolute_error(np.exp(y_train), y_pred_train)

test_RMSE = root_mean_squared_error(np.exp(y_test), y_pred_test)
train_RMSE = root_mean_squared_error(np.exp(y_train), y_pred_train)

test_r2 = r2_score(y_test, y_pred_test_log)
train_r2 = r2_score(y_train, y_pred_train_log)

print(f"Test MAE: {test_MAE:.2f}$ | Train MAE: {train_MAE:.2f}$")
print(f"Test RMSE: {test_RMSE:.2f}$ | Train RMSE: {train_RMSE:.2f}$")
print(f"Test R2 Score: {test_r2:.2f} | Train R2 Score: {train_r2:.2f}")

Test MAE: 53.55$ | Train MAE: 46.96$
Test RMSE: 121.80$ | Train RMSE: 108.13$
Test R2 Score: 0.67 | Train R2 Score: 0.74


In [10]:
results_df = pd.DataFrame(
    columns=['MAE', 'RMSE', 'R2']
)

results_df.loc['RandomForest (baseline, optimized)'] = [
    round(test_MAE, 2),
    round(test_RMSE, 2),
    round(test_r2, 2)
]

results_df

,MAE,RMSE,R2
"RandomForest (baseline, optimized)",53.55,121.8,0.67


## RandomForest. Hypyerparameter tuning Conclusion.

The primary objective of hyperparameter tuning was not to maximize predictive performance, but to reduce overfitting and obtain a more stable baseline model for comparison with other algorithms. This objective was sucessfully achieved by shifting the hyperparameter search toward stronger regularization, which significantly reduced the gap between the training and test performance. Although a moderate degree of overfitting still remains, it is acceptable for the purposes of this project and provides a reliable baseline for evaluating more advanced tree-based methods.

To avoid retraining during subsequent executions of the notebook, the optimized model was saved in the `./artifacts/` directory and can be loaded directly when needed. This reduce both computational cost and notebook execution time, while ensuring reproducible results.

# XGBoost

In this section, the **native XGBoost API** is used instead of the **scikit-learn** wrapper, providing greater flexibility over the training process. The approach enables the use of **GPU acceleration**, **Early Stopping** and seamless integration with **Optuna** by hyperparameter optimization.

Hyperparameter tuning is performed using **Optuna**, which employs an efficient search strategy to identify high-performing parameter configurations while requiring significantly fewer model evaluations than an exhaustive **Grid Search**. This allows the optimization process to be completed substantially faster without compromising model quality.

To ensure a fair comparison across all XGBoost experiments, the same optimization procedure, hyperparameter search space and training configuration are used for every model. The only difference between experiments is the set of input features (baseline, `zipcode`, `amenities`, `description embeddings` and thier combinations), allowing the impact of each feature set on the final predictive performance to be evaluated independently.

## XGBoost Baseline

The baseline XGBoost model is trained using the original feature set without `amenities`, `description embeddings` or `zipcode`. The model serves as a reference point for evaluating contribution of additional features in a subsequent experiments.

In [11]:
df_copy = builder.get_df(df, use_amenities=False, use_embeddings=False)

X = df_copy.drop(columns=['log_price'])
y = df_copy['log_price']

X = X.drop(columns=['zipcode'])

X.shape, y.shape

((74111, 25), (74111,))

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    random_state=42,
    test_size=0.2
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train,
    random_state=42,
    test_size=0.2
)

X_train.shape, X_test.shape, X_val.shape

((47430, 25), (14823, 25), (11858, 25))

In [13]:
cat_features = X_train.select_dtypes(include=['object', 'string']).columns

tree_preprocessor = preprocessor.create_tree_preprocessor(cat_features)

X_train_processed = tree_preprocessor.fit_transform(X_train, y_train)

X_val_processed = tree_preprocessor.transform(X_val)
X_test_processed = tree_preprocessor.transform(X_test)

X_train_processed = X_train_processed.astype(np.float32)
X_val_processed = X_val_processed.astype(np.float32)
X_test_processed = X_test_processed.astype(np.float32)

In [14]:
import xgboost as xgb
DEVICE = "cuda" if xgb.build_info()['USE_CUDA'] else "cpu"

if DEVICE == 'cuda':
    dtrain = xgb.QuantileDMatrix(X_train_processed, label=y_train)
    dval = xgb.QuantileDMatrix(X_val_processed, label=y_val, ref=dtrain)
    dtest = xgb.QuantileDMatrix(X_test_processed, label=y_test, ref=dtrain)
else:
    dtrain = xgb.DMatrix(X_train_processed, label=y_train)
    dval = xgb.DMatrix(X_val_processed, label=y_val)
    dtest = xgb.DMatrix(X_test_processed, label=y_test)

In [15]:
from optuna_integration import XGBoostPruningCallback
import xgboost as xgb

print(f"Using: {DEVICE} for objective funcion.")

def objective(trial):

    params = {
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "tree_method": "hist",
        "device": DEVICE,
        
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.08, log=True),
        "max_depth": trial.suggest_int("max_depth", 2, 6),
        "subsample": trial.suggest_float("subsample", 0.6, 0.9),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 0.8),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "min_child_weight": trial.suggest_int("min_child_weight", 5, 20),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 5, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1, 20, log=True),

        "verbosity": 0,
        "n_jobs": -1,
        "seed": 42
    }

    booster = xgb.train(
        params=params,
        dtrain=dtrain,
        num_boost_round=1000,
        evals=[(dval, "validation")],
        early_stopping_rounds=50,
        verbose_eval=False,
        callbacks=[XGBoostPruningCallback(trial, "validation-rmse")]
    )
    
    predictions = booster.predict(dval)
    rmse = root_mean_squared_error(y_val, predictions)
    
    return rmse

Using: cuda for objective funcion.


In [16]:
ARTIFACTS_DIR = Path('../artifacts')
ARTIFACTS_DIR.mkdir(exist_ok=True)

XGBOOST_BASELINE_PIPELINE = ARTIFACTS_DIR / "xgboost_baseline_pipeline"

In [17]:
%%time

import optuna
from utils.xgb_pipeline import XGBoostPipeline

if (
    XGBOOST_BASELINE_PIPELINE.with_suffix(".json").exists() and
    XGBOOST_BASELINE_PIPELINE.with_suffix(".joblib").exists() and
    XGBOOST_BASELINE_PIPELINE.with_suffix(".params").exists()
):
    print("Loading XGboost Baseline model...")
    xgboost_pipeline = XGBoostPipeline.load(XGBOOST_BASELINE_PIPELINE)
else:
    print("Training XGBoost Baseline model...")
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study = optuna.create_study(
        direction="minimize",
        pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=50, interval_steps=10)
    )
    
    study.optimize(objective, n_trials=250, gc_after_trial=True)

    best_params = {
        **study.best_params,
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "tree_method": "hist",
        "n_jobs": -1,
        "seed": 42,
        "verbosity": 0,
        "device": DEVICE
    }

    xgboost_pipeline = XGBoostPipeline(preprocessor=tree_preprocessor)
    xgboost_pipeline.fit(X_train, y_train, X_val, y_val, best_params)

    xgboost_pipeline.save(XGBOOST_BASELINE_PIPELINE)

xgboost_pipeline.params

Loading XGboost Baseline model...
CPU times: user 73.4 ms, sys: 1.02 ms, total: 74.4 ms
Wall time: 35.8 ms


{'learning_rate': 0.07991270587340771,
 'max_depth': 6,
 'subsample': 0.6989732057018632,
 'colsample_bytree': 0.7874022381003513,
 'gamma': 0.1542516218687972,
 'min_child_weight': 13,
 'reg_alpha': 0.006647249339056432,
 'reg_lambda': 6.240806241287537,
 'objective': 'reg:squarederror',
 'eval_metric': 'rmse',
 'tree_method': 'hist',
 'n_jobs': -1,
 'seed': 42,
 'verbosity': 0,
 'device': 'cuda'}

In [18]:
y_pred_test_log = xgboost_pipeline.predict(X_test)
y_pred_train_log = xgboost_pipeline.predict(X_train)

y_pred_test = np.exp(y_pred_test_log)
y_pred_train = np.exp(y_pred_train_log)

test_MAE = mean_absolute_error(np.exp(y_test), y_pred_test)
train_MAE = mean_absolute_error(np.exp(y_train), y_pred_train)

test_RMSE = root_mean_squared_error(np.exp(y_test), y_pred_test)
train_RMSE = root_mean_squared_error(np.exp(y_train), y_pred_train)

test_r2 = r2_score(y_test, y_pred_test_log)
train_r2 = r2_score(y_train, y_pred_train_log)

print(f"Test MAE: {test_MAE:.2f}$ | Train MAE: {train_MAE:.2f}$")
print(f"Test RMSE: {test_RMSE:.2f}$ | Train RMSE: {train_RMSE:.2f}$")
print(f"Test R2 Score: {test_r2:.2f} | Train R2 Score: {train_r2:.2f}")

Test MAE: 49.71$ | Train MAE: 42.12$
Test RMSE: 112.44$ | Train RMSE: 95.14$
Test R2 Score: 0.71 | Train R2 Score: 0.78


In [19]:
results_df.loc['XGBoost (baseline, optimized)'] = (
    round(test_MAE, 2),
    round(test_RMSE, 2),
    round(test_r2, 2)
)

results_df

,MAE,RMSE,R2
"RandomForest (baseline, optimized)",53.55,121.80,0.67
"XGBoost (baseline, optimized)",49.71,112.44,0.71


### Conclusion

The baseline XGBoost model outperformed a baseline RandomForest baseline model across all evaluation metrics. Compared to the RandomForest model, XGBosot reduced prediction error by approximately **$3.9 MAE** and improved coefficient of determination from **0.67** to **0.71**.

The difference between training and test metric remains relatively small, indicating only moderate overfitting and demonstrating good generalization on unseen data.

These results establish XGBoost as a stronger baseline model for the remaining experiments. Subsequent sections investigate whether incorporating additional information such as `zipcode`, `amenities` and `description embeddings` can further improve prediction performance.

## XGBoost Amenities + Description Embeddings

In [20]:
df_copy = builder.get_df(df, use_amenities=True, use_embeddings=True)

X = df_copy.drop(columns=['log_price'])
y = df_copy['log_price']

X = X.drop(columns=['zipcode'])

X.shape, y.shape

Loading embeddings from Parquet /home/carl/notebooks/airbnb_prices_prediction/dataset/description-embeddings.parquet...


((74111, 1166), (74111,))

In [21]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    random_state=42,
    test_size=0.2,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train,
    random_state=42,
    test_size=0.2
)

X_train.shape, X_val.shape, X_test.shape

((47430, 1166), (11858, 1166), (14823, 1166))

In [22]:
import xgboost as xgb

cat_features = X_train.select_dtypes(include=['object', 'string']).columns

tree_preprocessor = preprocessor.create_tree_preprocessor(cat_features)

X_train_processed = tree_preprocessor.fit_transform(X_train, y_train)
X_val_processed = tree_preprocessor.transform(X_val)
X_test_processed = tree_preprocessor.transform(X_test)

X_train_processed = X_train_processed.astype(np.float32)
X_val_processed = X_val_processed.astype(np.float32)
X_test_processed = X_test_processed.astype(np.float32)

if DEVICE == "cuda":
    dtrain = xgb.QuantileDMatrix(X_train_processed, label=y_train)
    dval = xgb.QuantileDMatrix(X_val_processed, label=y_val, ref=dtrain)
    dtest = xgb.QuantileDMatrix(X_test_processed, label=y_test, ref=dtrain)
else:
    dtrain = xgb.DMatrix(X_train_processed, label=y_train)
    dval = xgb.DMatrix(X_val_processed, label=y_val)
    dtest = xgb.DMatrix(X_test_processed, label=y_test)

In [23]:
ARTIFACTS_DIR = Path('../artifacts')
ARTIFACTS_DIR.mkdir(exist_ok=True)

XGBOOST_AMENITIES_EMBEDDINGS_PIPELINE = ARTIFACTS_DIR / "xgboost_amenities_embeddings_pipeline"

In [24]:
%%time

from utils.xgb_pipeline import XGBoostPipeline


if (
    XGBOOST_AMENITIES_EMBEDDINGS_PIPELINE.with_suffix(".json").exists() and
    XGBOOST_AMENITIES_EMBEDDINGS_PIPELINE.with_suffix(".joblib").exists() and
    XGBOOST_AMENITIES_EMBEDDINGS_PIPELINE.with_suffix(".params").exists()
):
    print("Loading XGboost (Full dataset) Pipeline...")
    xgboost_pipeline = XGBoostPipeline.load(XGBOOST_AMENITIES_EMBEDDINGS_PIPELINE)
else:
    print("Training XGBoost (Full dataset) Pipeline...")

    DEVICE = "cuda" if xgb.build_info()['USE_CUDA'] else "cpu"
    print(f"Training model, using device: {DEVICE}")
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study = optuna.create_study(
        direction="minimize",
        pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=50, interval_steps=10)
    )
    
    study.optimize(objective, n_trials=250, gc_after_trial=True)

    best_params = {
        **study.best_params,
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "tree_method": "hist",
        "n_jobs": -1,
        "seed": 42,
        "verbosity": 0,
        "device": DEVICE
    }

    xgboost_pipeline = XGBoostPipeline(preprocessor=tree_preprocessor)
    xgboost_pipeline.fit(
        X_train, y_train, X_val, y_val, best_params
    )

    xgboost_pipeline.save(XGBOOST_AMENITIES_EMBEDDINGS_PIPELINE)

xgboost_pipeline.params

Loading XGboost (Full dataset) Pipeline...
CPU times: user 94.4 ms, sys: 8 μs, total: 94.4 ms
Wall time: 54 ms


{'learning_rate': 0.057807097332236296,
 'max_depth': 6,
 'subsample': 0.8891124469741166,
 'colsample_bytree': 0.7808946084949187,
 'gamma': 0.06697537065233324,
 'min_child_weight': 15,
 'reg_alpha': 0.007673609836857946,
 'reg_lambda': 1.2791727550777485,
 'objective': 'reg:squarederror',
 'eval_metric': 'rmse',
 'tree_method': 'hist',
 'n_jobs': -1,
 'seed': 42,
 'verbosity': 0,
 'device': 'cuda'}

In [25]:
y_pred_test_log = xgboost_pipeline.predict(X_test)
y_pred_train_log = xgboost_pipeline.predict(X_train)

y_pred_test = np.exp(y_pred_test_log)
y_pred_train = np.exp(y_pred_train_log)

test_MAE = mean_absolute_error(np.exp(y_test), y_pred_test)
train_MAE = mean_absolute_error(np.exp(y_train), y_pred_train)

test_RMSE = root_mean_squared_error(np.exp(y_test), y_pred_test)
train_RMSE = root_mean_squared_error(np.exp(y_train), y_pred_train)

test_r2 = r2_score(y_test, y_pred_test_log)
train_r2 = r2_score(y_train, y_pred_train_log)

print(f"Test MAE: {test_MAE:.2f}$ | Train MAE: {train_MAE:.2f}$")
print(f"Test RMSE: {test_RMSE:.2f}$ | Train RMSE: {train_RMSE:.2f}$")
print(f"Test R2 Score: {test_r2:.2f} | Train R2 Score: {train_r2:.2f}")

Test MAE: 48.62$ | Train MAE: 24.59$
Test RMSE: 111.60$ | Train RMSE: 55.64$
Test R2 Score: 0.73 | Train R2 Score: 0.93


In [26]:
results_df.loc['XGBoost (amenities+embeddings, optimized)'] = (
    round(test_MAE, 2),
    round(test_RMSE, 2),
    round(test_r2, 2)
)

results_df

,MAE,RMSE,R2
"RandomForest (baseline, optimized)",53.55,121.80,0.67
"XGBoost (baseline, optimized)",49.71,112.44,0.71
"XGBoost (amenities+embeddings, optimized)",48.62,111.60,0.73


### Conclusion

The inclusion of `amenities` and `description embeddings` resulted in a modest improvement across the evaluation metrics. However, these additional features substantially increased the dimensionality of the dataset, leading to a training time that was approximately **2-4 times longer** than that of the baseline model.

Furthermore, the high dimensional embeddings representation (1024 features) was accompined by noticeably higher degree of overfitting, as reflected by a larger gap between training and test performance. While the model achieved slightly better predictive accuracy, the improvement was relatively small, compared to the additional computational cost and increased model complexity.

Overall, this experiment demonstrates that the limited performance gain does not justify the significantly longer training time and the increased overfitting, making the baseline XGBoost a more practical choice in terms of trade-off between predictive performance and computational efficiency.

## PCA For Description Embeddings

Before evaluating the impact of PCA on the XGBoost model, we first formulate a hypothesis that dimensionality reduction of description embeddings can improve overall training process without sacrificing prediction performance. The experiment is motivated by three primary objectives:

* Reduce training time by decreasing the number of description embeddings;
* Reduce overfitting by removing redundant and noisy dimensions;
* Preserve predictive performance, ensuring that dimensionality reduction does not degrade the model's accuracy;

To determine an appropriate number of principal components, we first analyze cumulative explained variance of the embedding space. Based on this analysis, several PCA configurations are evaluated and compared with the original 1024-dimensional embeddings to identify the best trade-off between dimensionality reduction, model generalization and predictive performance.

In [27]:
embeddings_df = builder.get_description_embeddings_df(df)

embeddings_df.shape

Loading embeddings from Parquet /home/carl/notebooks/airbnb_prices_prediction/dataset/description-embeddings.parquet...


(74111, 1024)

In [28]:
from sklearn.decomposition import PCA

pca = PCA()
pca.fit(embeddings_df)

explained_variance = np.cumsum(pca.explained_variance_ratio_)

explained_variance[:10]

array([0.04633991, 0.07908731, 0.10364504, 0.12456712, 0.14230436,
       0.15870565, 0.17409456, 0.18907511, 0.20272037, 0.21572368],
      dtype=float32)

In [29]:
for threshold in [0.90, 0.95, 0.96, 0.97, 0.99]:
    n_components = np.argmax(explained_variance >= threshold) + 1
    print(f"{threshold:.0%} variance: {n_components} components")

90% variance: 278 components
95% variance: 370 components
96% variance: 399 components
97% variance: 438 components
99% variance: 605 components


### N_Components 370

The first experiment uses **370 principal components**, which preserves approximately 95% of the total variance in the original 1024-dimensional embedding space. This threshold is commonly used in dimensionality reduction as it retains the vast majority of the information while substantially reducing the number of features. For this reason, it serves as a natural starting point for evaluating the effect of PCA on model performance and generalization.

In [30]:
pca = PCA(n_components=370, random_state=42)

embeddings_pca = pca.fit_transform(embeddings_df)

embeddings_pca.shape

(74111, 370)

In [31]:
embeddings_pca = pd.DataFrame(
    embeddings_pca,
    index=df_copy.index,
    columns=[f"embedding_pca_{i}" for i in range(embeddings_pca.shape[1])]
)

df_copy = builder.get_df(df, use_amenities=True, use_embeddings=False)

df_copy = pd.concat([df_copy, embeddings_pca], axis=1)

df_copy.shape

(74111, 514)

In [32]:
X = df_copy.drop(columns='log_price')
y = df_copy['log_price']

X = X.drop(columns=['zipcode'])

X.shape, y.shape

((74111, 512), (74111,))

In [33]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    random_state=42,
    test_size=0.2
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train,
    random_state=42,
    test_size=0.2
)

X_train.shape, X_test.shape, X_val.shape

((47430, 512), (14823, 512), (11858, 512))

In [34]:
cat_features = X_train.select_dtypes(include=['object', 'string']).columns

tree_preprocessor = preprocessor.create_tree_preprocessor(cat_features)

params = xgboost_pipeline.params

pca_pipeline = XGBoostPipeline(
    preprocessor=tree_preprocessor
)

pca_pipeline.fit(X_train, y_train, X_val, y_val, params)

y_pred_test_log = pca_pipeline.predict(X_test)
y_pred_train_log = pca_pipeline.predict(X_train)

y_pred_test = np.exp(y_pred_test_log)
y_pred_train = np.exp(y_pred_train_log)

test_MAE = mean_absolute_error(np.exp(y_test), y_pred_test)
train_MAE = mean_absolute_error(np.exp(y_train), y_pred_train)

test_RMSE = root_mean_squared_error(np.exp(y_test), y_pred_test)
train_RMSE = root_mean_squared_error(np.exp(y_train), y_pred_train)

test_r2 = r2_score(y_test, y_pred_test_log)
train_r2 = r2_score(y_train, y_pred_train_log)

print(f"Test MAE: {test_MAE:.2f}$ | Train MAE: {train_MAE:.2f}$")
print(f"Test RMSE: {test_RMSE:.2f}$ | Train RMSE: {train_RMSE:.2f}$")
print(f"Test R2 Score: {test_r2:.2f} | Train R2 Score: {train_r2:.2f}")

Test MAE: 48.41$ | Train MAE: 31.43$
Test RMSE: 110.88$ | Train RMSE: 70.58$
Test R2 Score: 0.73 | Train R2 Score: 0.89


The original model with 1024-dimensional embedding space:

Test MAE: 48.62 | Train MAE: 24.59

Test RMSE: 111.60 | Train RMSE: 55.64

Test R2 Score: 0.73 | Train R2 Score: 0.93

#### Conclusion 

Applying PCA with **370 principal components**, produced encouraging results. The gap between training and test metric became noticeably smaller, indicating a reduction in overfitting compared to the original 1024-dimensional embeddings. At the same time, the model preserved its predictive performance, with the test metrics showing a slight improvement over the original model.

This results suggest that a significant portion of the embedding dimensions is redundant for the XGBoost model. As a result, the next experiment investigates a more aggresive dimensionality reduction of **278 principal components**, which preserve **90% of the total explained variance**, to determine whether further reducing the embedding space can improve generalization without sacrificing predictive performance.  

### N_Components 278

In [35]:
embeddings_df = builder.get_description_embeddings_df(df)

pca = PCA(n_components=278, random_state=42)

embeddings_pca = pca.fit_transform(embeddings_df)

embeddings_pca.shape

Loading embeddings from Parquet /home/carl/notebooks/airbnb_prices_prediction/dataset/description-embeddings.parquet...


(74111, 278)

In [36]:
embeddings_pca = pd.DataFrame(
    embeddings_pca,
    index=df_copy.index,
    columns=[f"embedding_pca_{i}" for i in range(embeddings_pca.shape[1])]
)

df_copy = builder.get_df(df, use_amenities=True, use_embeddings=False)

df_copy = pd.concat([df_copy, embeddings_pca], axis=1)

df_copy.shape

(74111, 422)

In [37]:
X = df_copy.drop(columns='log_price')
y = df_copy['log_price']

X = X.drop(columns=['zipcode'])

X.shape, y.shape

((74111, 420), (74111,))

In [38]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    random_state=42,
    test_size=0.2
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train,
    random_state=42,
    test_size=0.2
)

X_train.shape, X_test.shape, X_val.shape

((47430, 420), (14823, 420), (11858, 420))

In [39]:
cat_features = X_train.select_dtypes(include=['object', 'string']).columns

tree_preprocessor = preprocessor.create_tree_preprocessor(cat_features)

params = xgboost_pipeline.params

pca_pipeline = XGBoostPipeline(
    preprocessor=tree_preprocessor
)

pca_pipeline.fit(X_train, y_train, X_val, y_val, params)

y_pred_test_log = pca_pipeline.predict(X_test)
y_pred_train_log = pca_pipeline.predict(X_train)

y_pred_test = np.exp(y_pred_test_log)
y_pred_train = np.exp(y_pred_train_log)

test_MAE = mean_absolute_error(np.exp(y_test), y_pred_test)
train_MAE = mean_absolute_error(np.exp(y_train), y_pred_train)

test_RMSE = root_mean_squared_error(np.exp(y_test), y_pred_test)
train_RMSE = root_mean_squared_error(np.exp(y_train), y_pred_train)

test_r2 = r2_score(y_test, y_pred_test_log)
train_r2 = r2_score(y_train, y_pred_train_log)

print(f"Test MAE: {test_MAE:.2f}$ | Train MAE: {train_MAE:.2f}$")
print(f"Test RMSE: {test_RMSE:.2f}$ | Train RMSE: {train_RMSE:.2f}$")
print(f"Test R2 Score: {test_r2:.2f} | Train R2 Score: {train_r2:.2f}")

Test MAE: 47.91$ | Train MAE: 28.16$
Test RMSE: 110.25$ | Train RMSE: 64.03$
Test R2 Score: 0.73 | Train R2 Score: 0.91


The original model with 1024-dimensional embedding space:

Test MAE: 48.62 | Train MAE: 24.59

Test RMSE: 111.60 | Train RMSE: 55.64

Test R2 Score: 0.73 | Train R2 Score: 0.93

#### Conclusion 

Using **278 principal components** further reduced the embedding dimensionality while preserving **90% of the total explained variance**. Compared to the originall 1024-dimensional embeddings, the gap between the training and test metrics remained noticeably smaller, confirming that PCA continued to navigate overfitting. However, the reduction was in overfitting was not as pronounced as in the previous experiment with 370 components.

Intrestingly, the configuration achieved the best predictive performance among all evaluated embeddings representations, slightly outperforming both the original 1024-dimensional embeddings and the 370-component PCA model. However, the primiary objective of this study was not to maximize the predictive performance, but to identify a dimensionality reduction strategy that effectively reduces overfitting while preserving model quality. From this perspective, the **370-component configuration** provides a better balance between generalization and predictive performance, making it the perfect choice for subsequent experiments.

## PCA For Description Embeddings. Conclusion

Based on the conducted experiments, `n_components=370` **was selected as the best trade-off** between dimensionality reduction, model generalization and preservation of predictive performance.

To improve reusability and flexibility of the feature engineering pipeline, the `FeatureBuilder()` class was updated (`./preprocessing/features.py`). A new parameter, `embedding_pca_components` was intoduced, allowing optional PCA transformation for description embeddings during feature generation.

The default value is set to `None`, meaning that the original embeddings will be used unless dimensionality reduction explicitly requested.

Example usage:

    builder = FeatureBuilder()
    df_copy = builder.get_df(df, use_embeddings=True, embedding_pca_components=370)

This approach allows the same feature engineering pipeline to support both original embeddings and PCA-reduced representations without requiring additional preprocessing logic.

## XGBoost Baseline + Zipcode

In [40]:
df_copy = builder.get_df(df, use_amenities=False, use_embeddings=False)

X = df_copy.drop(columns=['log_price'])
y = df_copy['log_price']

X.shape, y.shape

((74111, 26), (74111,))

In [41]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    random_state=42,
    test_size=0.2
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train,
    random_state=42,
    test_size=0.2
)

X_train.shape, X_val.shape, X_test.shape

((47430, 26), (11858, 26), (14823, 26))

In [42]:
cat_features = X_train.select_dtypes(include=['object', 'string']).columns

tree_preprocessor = preprocessor.create_tree_preprocessor(cat_features)

X_train_processed = tree_preprocessor.fit_transform(X_train, y_train)
X_val_processed = tree_preprocessor.transform(X_val)
X_test_processed = tree_preprocessor.transform(X_test)

X_train_processed = X_train_processed.astype(np.float32)
X_val_processed = X_val_processed.astype(np.float32)
X_test_processed = X_test_processed.astype(np.float32)

DEVICE = 'cuda' if xgb.build_info()['USE_CUDA'] else 'cpu'
print(f"Currently using device is: {DEVICE}")

if DEVICE == 'cuda':
    dtrain = xgb.QuantileDMatrix(X_train_processed, label=y_train)
    dval = xgb.QuantileDMatrix(X_val_processed, label=y_val, ref=dtrain)
    dtest = xgb.QuantileDMatrix(X_test_processed, label=y_test, ref=dtrain)
else:
    dtrain = xgb.DMatrix(X_train_processed, label=y_train)
    dval = xgb.DMatrix(X_val_processed, label=y_val)
    dtest = xgb.DMatrix(X_test_processed, label=y_test)

Currently using device is: cuda


In [43]:
ARTIFACTS_DIR = Path('../artifacts')
ARTIFACTS_DIR.mkdir(exist_ok=True)

XGBOOST_BASELINE_ZIPCODE = ARTIFACTS_DIR / "xgboost_baseline_zipcode"

In [44]:
%%time

if (
    XGBOOST_BASELINE_ZIPCODE.with_suffix(".json").exists() and
    XGBOOST_BASELINE_ZIPCODE.with_suffix(".joblib").exists() and 
    XGBOOST_BASELINE_ZIPCODE.with_suffix(".params").exists()
):
    print("Loading XGBoost Baseline Model with zipcode...")
    xgboost_pipeline = XGBoostPipeline.load(XGBOOST_BASELINE_ZIPCODE)
else:
    print("Training XGBoost Baseline Model with zipcode...")
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    study = optuna.create_study(
        direction="minimize",
        pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=50, interval_steps=10)
    )

    study.optimize(objective, n_trials=250, gc_after_trial=True)

    best_params = {
        **study.best_params,
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "tree_method": "hist",
        "n_jobs": -1,
        "seed": 42,
        "verbosity": 0,
        "device": DEVICE
    }
    xgboost_pipeline = XGBoostPipeline(preprocessor=tree_preprocessor)
    xgboost_pipeline.fit(
        X_train, y_train,
        X_val, y_val,
        params=best_params
    )

    xgboost_pipeline.save(XGBOOST_BASELINE_ZIPCODE)

xgboost_pipeline.params

Loading XGBoost Baseline Model with zipcode...
CPU times: user 73.3 ms, sys: 14 μs, total: 73.3 ms
Wall time: 32.2 ms


{'learning_rate': 0.0798368989397859,
 'max_depth': 6,
 'subsample': 0.7905213385920479,
 'colsample_bytree': 0.5767538238858704,
 'gamma': 0.17029931745181887,
 'min_child_weight': 16,
 'reg_alpha': 0.0019212395603417612,
 'reg_lambda': 2.0602095326590844,
 'objective': 'reg:squarederror',
 'eval_metric': 'rmse',
 'tree_method': 'hist',
 'n_jobs': -1,
 'seed': 42,
 'verbosity': 0,
 'device': 'cuda'}

In [45]:
y_pred_test_log = xgboost_pipeline.predict(X_test)
y_pred_train_log = xgboost_pipeline.predict(X_train)

y_pred_test = np.exp(y_pred_test_log)
y_pred_train = np.exp(y_pred_train_log)

test_MAE = mean_absolute_error(np.exp(y_test), y_pred_test)
train_MAE = mean_absolute_error(np.exp(y_train), y_pred_train)

test_RMSE = root_mean_squared_error(np.exp(y_test), y_pred_test)
train_RMSE = root_mean_squared_error(np.exp(y_train), y_pred_train)

test_r2 = r2_score(y_test, y_pred_test_log)
train_r2 = r2_score(y_train, y_pred_train_log)

print(f"Test MAE: {test_MAE:.2f}$ | Train MAE: {train_MAE:.2f}$")
print(f"Test RMSE: {test_RMSE:.2f}$ | Train RMSE: {train_RMSE:.2f}$")
print(f"Test R2 Score: {test_r2:.2f} | Train R2 Score: {train_r2:.2f}")

Test MAE: 49.34$ | Train MAE: 42.77$
Test RMSE: 112.71$ | Train RMSE: 97.14$
Test R2 Score: 0.72 | Train R2 Score: 0.78


In [46]:
results_df.loc['XGBoost (baseline, optimized, zipcode)'] = (
    round(test_MAE, 2),
    round(test_RMSE, 2),
    round(test_r2, 2)
)

results_df.sort_values(by='RMSE', ascending=False)

,MAE,RMSE,R2
"RandomForest (baseline, optimized)",53.55,121.80,0.67
"XGBoost (baseline, optimized, zipcode)",49.34,112.71,0.72
"XGBoost (baseline, optimized)",49.71,112.44,0.71
"XGBoost (amenities+embeddings, optimized)",48.62,111.60,0.73


### Conclusion

Adding the **zipcode** feature resulted in only a marginal change in the model's performance. While R2 Score increased slightly, the overall predictive performance remained the nearly identical to the baseline model, with no meaningful improvement in the primary evaluation metric (RMSE). The training and test metrics also remained as a similar level, indicating that the additional information did not noticeably affect the model's generalization.

Overall, the **zipcode** feature provides only a limited contribution when used with the baseline feature set, suggesting that the existing numerical and categorical features already capture most of the available location-related information.

## XGBoost Amenities, Description Embeddings + Zipcode

In [47]:
df_copy = builder.get_df(df, use_amenities=True, use_embeddings=True, embedding_pca_components=370)

X = df_copy.drop(columns=['log_price'])
y = df_copy['log_price']

X.shape, y.shape

Loading embeddings from Parquet /home/carl/notebooks/airbnb_prices_prediction/dataset/description-embeddings.parquet...
Applying PCA (370 components)...
Embeddings shape is: (74111, 370)


((74111, 513), (74111,))

In [48]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    random_state=42,
    test_size=0.2
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train,
    random_state=42,
    test_size=0.2
)

X_train.shape, X_val.shape, X_test.shape

((47430, 513), (11858, 513), (14823, 513))

In [49]:
cat_features = X_train.select_dtypes(include=['object', 'string']).columns

tree_preprocessor = preprocessor.create_tree_preprocessor(cat_features)

X_train_processed = tree_preprocessor.fit_transform(X_train, y_train)
X_val_processed = tree_preprocessor.transform(X_val)
X_test_processed = tree_preprocessor.transform(X_test)

X_train_processed = X_train_processed.astype(np.float32)
X_val_processed = X_val_processed.astype(np.float32)
X_test_processed = X_test_processed.astype(np.float32)

DEVICE = 'cuda' if xgb.build_info()['USE_CUDA'] else 'cpu'
print(f"Currently using device is: {DEVICE}")

if DEVICE == 'cuda':
    dtrain = xgb.QuantileDMatrix(X_train_processed, label=y_train)
    dval = xgb.QuantileDMatrix(X_val_processed, label=y_val, ref=dtrain)
    dtest = xgb.QuantileDMatrix(X_test_processed, label=y_test, ref=dtrain)
else:
    dtrain = xgb.DMatrix(X_train_processed, label=y_train)
    dval = xgb.DMatrix(X_val_processed, label=y_val)
    dtest = xgb.DMatrix(X_test_processed, label=y_test)

Currently using device is: cuda


In [50]:
ARTIFACTS_DIR = Path('../artifacts')
ARTIFACTS_DIR.mkdir(exist_ok=True)

XGBOOST_AM_EM_ZIPCODE = ARTIFACTS_DIR / "xgboost_am_em_zipcode"

In [51]:
%%time

if (
    XGBOOST_AM_EM_ZIPCODE.with_suffix(".json").exists() and
    XGBOOST_AM_EM_ZIPCODE.with_suffix(".joblib").exists() and
    XGBOOST_AM_EM_ZIPCODE.with_suffix(".params").exists()
):
    print("Loading XGBoost Model with amenities, embeddings and zipcode...")
    xgboost_pipeline = XGBoostPipeline.load(XGBOOST_AM_EM_ZIPCODE)
else:
    print("Training XGBoost Model with amenities, embeddings and zipcode...")
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    study = optuna.create_study(
        direction="minimize",
        pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=50, interval_steps=10)
    )

    study.optimize(objective, n_trials=250, gc_after_trial=True)

    best_params = {
        **study.best_params,
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "tree_method": "hist",
        "n_jobs": -1,
        "seed": 42,
        "verbosity": 0,
        "device": DEVICE
    }

    xgboost_pipeline = XGBoostPipeline(preprocessor=tree_preprocessor)
    xgboost_pipeline.fit(
        X_train, y_train,
        X_val, y_val,
        params=best_params
    )

    xgboost_pipeline.save(XGBOOST_AM_EM_ZIPCODE)

xgboost_pipeline.params

Loading XGBoost Model with amenities, embeddings and zipcode...
CPU times: user 92.2 ms, sys: 0 ns, total: 92.2 ms
Wall time: 49.3 ms


{'learning_rate': 0.07322366483934734,
 'max_depth': 6,
 'subsample': 0.8849159180303793,
 'colsample_bytree': 0.6553328410431222,
 'gamma': 0.1275850155761123,
 'min_child_weight': 6,
 'reg_alpha': 0.030508360498301645,
 'reg_lambda': 2.1237416589735774,
 'objective': 'reg:squarederror',
 'eval_metric': 'rmse',
 'tree_method': 'hist',
 'n_jobs': -1,
 'seed': 42,
 'verbosity': 0,
 'device': 'cuda'}

In [52]:
y_pred_test_log = xgboost_pipeline.predict(X_test)
y_pred_train_log = xgboost_pipeline.predict(X_train)

y_pred_test = np.exp(y_pred_test_log)
y_pred_train = np.exp(y_pred_train_log)

test_MAE = mean_absolute_error(np.exp(y_test), y_pred_test)
train_MAE = mean_absolute_error(np.exp(y_train), y_pred_train)

test_RMSE = root_mean_squared_error(np.exp(y_test), y_pred_test)
train_RMSE = root_mean_squared_error(np.exp(y_train), y_pred_train)

test_r2 = r2_score(y_test, y_pred_test_log)
train_r2 = r2_score(y_train, y_pred_train_log)

print(f"Test MAE: {test_MAE:.2f}$ | Train MAE: {train_MAE:.2f}$")
print(f"Test RMSE: {test_RMSE:.2f}$ | Train RMSE: {train_RMSE:.2f}$")
print(f"Test R2 Score: {test_r2:.2f} | Train R2 Score: {train_r2:.2f}")

Test MAE: 48.06$ | Train MAE: 26.02$
Test RMSE: 110.91$ | Train RMSE: 58.01$
Test R2 Score: 0.73 | Train R2 Score: 0.92


In [55]:
results_df.loc['XGBoost (amenities, embeddings, optimized, zipcode, pca_components=370)'] = (
    round(test_MAE, 2),
    round(test_RMSE, 2),
    round(test_r2, 2)
)

results_df.sort_values(by='RMSE', ascending=False)

,MAE,RMSE,R2
"RandomForest (baseline, optimized)",53.55,121.80,0.67
"XGBoost (baseline, optimized, zipcode)",49.34,112.71,0.72
"XGBoost (baseline, optimized)",49.71,112.44,0.71
"XGBoost (amenities+embeddings, optimized)",48.62,111.60,0.73
"XGBoost (amenities, embeddings, optimized, zipcode)",48.06,110.91,0.73
"XGBoost (amenities, embeddings, optimized, zipcode, embedding_pca_components=370)",48.06,110.91,0.73
"XGBoost (amenities, embeddings, optimized, zipcode, pca_components=370)",48.06,110.91,0.73


### Conclusion

Adding the `zipcode` feature to the full feature set resulted in a small improvement in predictive performance, producing the best overall test metrics among the evaluated XGBoost models. However, the improvement came at the cost of increased overfitting, as the gap between the training and test metrics became noticeably larger compared to the PCA-based model without the zipcode feature. 

These results suggest that while **zipcode** provides additional location-specific information, that benefits prediction accuracy, it also increases the model's tendency to memorize the training data. Overall, the improvement in predictive performance is relatively modest, making the trade-off between accuracy and generalization an important consideration.

# Overfitting Reduction

The next stage is focused on reducing the remaning overfitting, while preserving the predictive performance of the model. To establish the reliable starting point, a new XGBoost model will be trained from scratch using the same optimization strategy as in previous experiments. The only difference is input dataset, which now includes the PCA-reduced description embeddings (`embedding_pca_components=370`).

The optimized PCA-based model, will serve as reference model for all subsequent experiments. The first objective is to analyze the remaining overfitting, identify the features or modeling choices that may contribute to it and formulate new hypotheses. Based on these findings, additional experiments will be conducted to improve the model's generalization without sacrificing predictive performance.

In [56]:
EMBEDDING_PCA_COMPONENTS = 370

df_copy = builder.get_df(df, use_amenities=True, use_embeddings=True, embedding_pca_components=EMBEDDING_PCA_COMPONENTS)

X = df_copy.drop(columns=['log_price'])
y = df_copy['log_price']

X = X.drop(columns=['zipcode'])

X.shape, y.shape

Loading embeddings from Parquet /home/carl/notebooks/airbnb_prices_prediction/dataset/description-embeddings.parquet...
Applying PCA (370 components)...
Embeddings shape is: (74111, 370)


((74111, 512), (74111,))

In [57]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    random_state=42,
    test_size=0.2
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train,
    random_state=42,
    test_size=0.2
)

X_train.shape, X_val.shape, X_test.shape

((47430, 512), (11858, 512), (14823, 512))

In [58]:
cat_features = X_train.select_dtypes(include=['object', 'string']).columns

tree_preprocessor = preprocessor.create_tree_preprocessor(cat_features)

X_train_processed = tree_preprocessor.fit_transform(X_train, y_train)
X_val_processed = tree_preprocessor.transform(X_val)
X_test_processed = tree_preprocessor.transform(X_test)

X_train_processed = X_train_processed.astype(np.float32)
X_val_processed = X_val_processed.astype(np.float32)
X_test_processed = X_test_processed.astype(np.float32)

DEVICE = 'cuda' if xgb.build_info()['USE_CUDA'] else 'cpu'
print(f"Currently using device: {DEVICE}")

if DEVICE == 'cuda':
    dtrain = xgb.QuantileDMatrix(X_train_processed, label=y_train)
    dval = xgb.QuantileDMatrix(X_val_processed, label=y_val, ref=dtrain)
    dtest = xgb.QuantileDMatrix(X_test_processed, label=y_test, ref=dtrain)
else:
    dtrain = xgb.DMatrix(X_train_processed, label=y_train)
    dval = xgb.DMatrix(X_val_processed, label=y_val)
    dtest = xgb.DMatrix(X_test_processed, label=y_test)
    

Currently using device: cuda


In [59]:
ARTIFACTS_DIR = Path('../artifacts')
ARTIFACTS_DIR.mkdir(exist_ok=True)

XGBOOST_REFERENCE = ARTIFACTS_DIR / "xgboost_reference"

In [60]:
%%time

from utils.xgb_pipeline import XGBoostPipeline


if (
    XGBOOST_REFERENCE.with_suffix(".json").exists() and
    XGBOOST_REFERENCE.with_suffix(".joblib").exists() and
    XGBOOST_REFERENCE.with_suffix(".params").exists()
):
    print("Loading XGBoost reference model...")
    xgboost_pipeline = XGBoostPipeline.load(XGBOOST_REFERENCE)
else:
    print("Training XGBoost reference model...")
    
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study = optuna.create_study(
        direction="minimize",
        pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=50, interval_steps=10)
    )
    
    study.optimize(objective, n_trials=250, gc_after_trial=True)

    best_params = {
        **study.best_params,
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "tree_method": "hist",
        "n_jobs": -1,
        "seed": 42,
        "verbosity": 0,
        "device": DEVICE
    }

    xgboost_pipeline = XGBoostPipeline(preprocessor=tree_preprocessor)
    xgboost_pipeline.fit(
        X_train, y_train, X_val, y_val, best_params
    )

    xgboost_pipeline.save(XGBOOST_REFERENCE)

xgboost_pipeline.params

Training XGBoost reference model...
CPU times: user 11min 41s, sys: 2.61 s, total: 11min 44s
Wall time: 11min 32s


{'learning_rate': 0.05865354326265577,
 'max_depth': 6,
 'subsample': 0.8781354172443845,
 'colsample_bytree': 0.6397027221367817,
 'gamma': 0.2431456358186981,
 'min_child_weight': 20,
 'reg_alpha': 0.006679521564569216,
 'reg_lambda': 7.483898889302057,
 'objective': 'reg:squarederror',
 'eval_metric': 'rmse',
 'tree_method': 'hist',
 'n_jobs': -1,
 'seed': 42,
 'verbosity': 0,
 'device': 'cuda'}

In [61]:
y_pred_test_log = xgboost_pipeline.predict(X_test)
y_pred_train_log = xgboost_pipeline.predict(X_train)

y_pred_test = np.exp(y_pred_test_log)
y_pred_train = np.exp(y_pred_train_log)

test_MAE = mean_absolute_error(np.exp(y_test), y_pred_test)
train_MAE = mean_absolute_error(np.exp(y_train), y_pred_train)

test_RMSE = root_mean_squared_error(np.exp(y_test), y_pred_test)
train_RMSE = root_mean_squared_error(np.exp(y_train), y_pred_train)

test_r2 = r2_score(y_test, y_pred_test_log)
train_r2 = r2_score(y_train, y_pred_train_log)

print(f"Test MAE: {test_MAE:.2f}$ | Train MAE: {train_MAE:.2f}$")
print(f"Test RMSE: {test_RMSE:.2f}$ | Train RMSE: {train_RMSE:.2f}$")
print(f"Test R2 Score: {test_r2:.2f} | Train R2 Score: {train_r2:.2f}")

Test MAE: 48.25$ | Train MAE: 28.63$
Test RMSE: 111.09$ | Train RMSE: 66.19$
Test R2 Score: 0.73 | Train R2 Score: 0.91
